# 1단계: 차량 탐지 모델 (COCO 사전학습 → AI-Hub 165 파인튜닝)

- 데이터: AI-Hub "교통문제 해결을 위한 CCTV 교통 영상(시내도로)" (dataSetSn=165)
- 방식: COCO 사전학습 YOLO 가중치 → 165번 데이터로 파인튜닝
- 클래스: 승용차, 소형버스, 대형버스, 트럭, 대형트레일러, 오토바이, 보행자 (7종)

> ⚠️ Cell 5(라벨 구조 확인)의 출력 결과에 따라 Cell 7(`parse_label`)의 키 이름을
> 실제 값으로 한 번 조정해야 할 수 있습니다. AI-Hub 데이터셋마다 JSON 필드명이
> 조금씩 다르기 때문입니다.


## 0. 환경 확인 (GPU / 패키지)

In [1]:
import os, sys, json, shutil, random
from pathlib import Path

import cv2
import torch
from tqdm import tqdm

print("Python:", sys.version)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)


Python: 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
CUDA available: True
GPU: NVIDIA GeForce RTX 5060 Laptop GPU
CUDA version: 12.8


In [4]:
# 처음 한 번만 실행 (이미 설치되어 있으면 생략 가능)
# !pip install ultralytics scikit-learn

from ultralytics import YOLO


## 1. 데이터 경로 설정

AI-Hub에서 받은 데이터는 보통 `원천데이터`(이미지)와 `라벨링데이터`(json)가
분리되어 있고, 그 아래로 촬영 지점/날짜/시간대별로 여러 단계의 하위 폴더가
있습니다. **최상위 폴더 경로만 지정**하면 아래 함수가 몇 단계든 하위 폴더를
전부 타고 들어가서 이미지/라벨 파일을 전부 찾아줍니다.

In [6]:
# ==== 여기 두 줄만 실제 경로로 수정하세요 ====
IMAGE_ROOT = Path(r"C:\Users\Win11Pro\Downloads\교통문제 해결을 위한 CCTV 교통 영상(시내도로)\Training")   # 이미지 최상위 폴더
LABEL_ROOT = Path(r"C:\Users\Win11Pro\Downloads\101.교통문제 해결을 위한 CCTV 교통 데이터(시내도로)\01.데이터\1.Training")  # 라벨(json) 최상위 폴더
# ============================================

IMAGE_EXTS = {".jpg", ".jpeg", ".png"}

def find_all_files(root: Path, exts: set):
    """root 아래 모든 하위 폴더(깊이 상관없이)를 재귀적으로 탐색해서
    지정한 확장자의 파일을 전부 리스트로 반환한다."""
    if not root.exists():
        raise FileNotFoundError(f"경로가 존재하지 않습니다: {root}")
    return [p for p in root.rglob("*") if p.is_file() and p.suffix.lower() in exts]

image_files = find_all_files(IMAGE_ROOT, IMAGE_EXTS)
label_files = find_all_files(LABEL_ROOT, {".json"})

print(f"이미지 파일: {len(image_files):,}개")
print(f"라벨 파일:   {len(label_files):,}개")
print("이미지 예시:", image_files[0] if image_files else "없음")
print("라벨 예시:  ", label_files[0] if label_files else "없음")


이미지 파일: 456,584개
라벨 파일:   137개
이미지 예시: C:\Users\Win11Pro\Downloads\교통문제 해결을 위한 CCTV 교통 영상(시내도로)\Training\교통안전(Bbox)\[원천]LG전자 부천랜드점 부근\LG전자 부천랜드점 부근\BC2000201\20201014_015958_S_1089.jpg
라벨 예시:   C:\Users\Win11Pro\Downloads\101.교통문제 해결을 위한 CCTV 교통 데이터(시내도로)\01.데이터\1.Training\교통안전(Bbox)\1.라벨링데이터_230510_add\1.라벨링데이터\LG전자 부천랜드점 부근\BC2000201\LG전자 부천랜드점 부근_BC2000201.json


## 2. 이미지 ↔ 라벨 매칭

AI-Hub 데이터는 보통 이미지와 라벨의 **파일명(확장자 제외)이 동일**합니다.
(폴더 구조는 서로 달라도 상관없이 파일명 기준으로 매칭합니다.)

In [7]:
label_lookup = {p.stem: p for p in label_files}

pairs = []       # (이미지 경로, 라벨 경로)
missing = []      # 라벨을 못 찾은 이미지

for img_path in image_files:
    lbl_path = label_lookup.get(img_path.stem)
    if lbl_path is not None:
        pairs.append((img_path, lbl_path))
    else:
        missing.append(img_path)

print(f"매칭된 쌍: {len(pairs):,}개")
print(f"라벨 없는 이미지: {len(missing):,}개")
if missing:
    print("예시 (최대 5개):")
    for m in missing[:5]:
        print(" -", m)


매칭된 쌍: 0개
라벨 없는 이미지: 456,584개
예시 (최대 5개):
 - C:\Users\Win11Pro\Downloads\교통문제 해결을 위한 CCTV 교통 영상(시내도로)\Training\교통안전(Bbox)\[원천]LG전자 부천랜드점 부근\LG전자 부천랜드점 부근\BC2000201\20201014_015958_S_1089.jpg
 - C:\Users\Win11Pro\Downloads\교통문제 해결을 위한 CCTV 교통 영상(시내도로)\Training\교통안전(Bbox)\[원천]LG전자 부천랜드점 부근\LG전자 부천랜드점 부근\BC2000201\20201014_015958_S_1287.jpg
 - C:\Users\Win11Pro\Downloads\교통문제 해결을 위한 CCTV 교통 영상(시내도로)\Training\교통안전(Bbox)\[원천]LG전자 부천랜드점 부근\LG전자 부천랜드점 부근\BC2000201\20201014_015958_S_1386.jpg
 - C:\Users\Win11Pro\Downloads\교통문제 해결을 위한 CCTV 교통 영상(시내도로)\Training\교통안전(Bbox)\[원천]LG전자 부천랜드점 부근\LG전자 부천랜드점 부근\BC2000201\20201014_015958_S_1485.jpg
 - C:\Users\Win11Pro\Downloads\교통문제 해결을 위한 CCTV 교통 영상(시내도로)\Training\교통안전(Bbox)\[원천]LG전자 부천랜드점 부근\LG전자 부천랜드점 부근\BC2000201\20201014_015958_S_1683.jpg


## 3. 라벨 JSON 구조 확인 (중요)

파싱 함수를 만들기 전에, 실제 라벨 파일이 어떤 구조인지 먼저 눈으로 확인합니다.
**이 셀의 출력을 보고 Cell 7의 `parse_label` 함수에서 표시된 부분(`# <-- 조정`)만
실제 키 이름으로 바꿔주시면 됩니다.**

In [ ]:
sample_img, sample_lbl = pairs[0]
with open(sample_lbl, encoding="utf-8") as f:
    sample = json.load(f)

print("최상위 키:", list(sample.keys()))
print("-" * 50)
print(json.dumps(sample, indent=2, ensure_ascii=False)[:3000])


## 4. 클래스 매핑 (165번 데이터셋 차종분류 7종)

In [ ]:
CLASS_MAP = {
    "승용차": 0,
    "소형버스": 1,
    "대형버스": 2,
    "트럭": 3,
    "대형트레일러": 4,
    "오토바이": 5,
    "보행자": 6,
}
CLASS_NAMES = list(CLASS_MAP.keys())
print(CLASS_MAP)


## 5. 라벨 파싱 함수

⚠️ 아래는 AI-Hub에서 흔히 쓰이는 몇 가지 포맷을 가정한 기본 구현입니다.
Cell 5(구조 확인)의 실제 출력과 다르면 `# <-- 조정` 표시된 줄만 실제 키 이름으로
바꿔주세요. (출력 내용을 저에게 보여주시면 정확히 맞춰드릴 수 있습니다.)

In [ ]:
def parse_label(json_path: Path, img_w: int, img_h: int):
    """AI-Hub 라벨 json -> [(class_id, x_center, y_center, w, h), ...]
    좌표는 YOLO 포맷(0~1 정규화)으로 반환한다.
    """
    with open(json_path, encoding="utf-8") as f:
        data = json.load(f)

    # 어노테이션 리스트 위치: 데이터셋마다 다름  # <-- 조정
    annos = (
        data.get("annotations")
        or data.get("objects")
        or data.get("Learning_Data_Info", {}).get("Annotations")
    )
    if annos is None:
        raise KeyError(f"어노테이션 키를 찾지 못했습니다. 최상위 키: {list(data.keys())}")

    results = []
    for obj in annos:
        # 클래스명이 들어있는 키  # <-- 조정
        cls_name = obj.get("class_name") or obj.get("category") or obj.get("class")
        # bbox가 들어있는 키, 보통 [x, y, w, h] (좌상단 + 너비/높이)  # <-- 조정
        bbox = obj.get("bbox") or obj.get("box")

        if cls_name not in CLASS_MAP or bbox is None or len(bbox) != 4:
            continue

        x, y, w, h = bbox
        # bbox가 [xmin, ymin, xmax, ymax] 형태라면 아래 두 줄 주석을 해제하세요
        # w = w - x
        # h = h - y

        xc = (x + w / 2) / img_w
        yc = (y + h / 2) / img_h
        nw = w / img_w
        nh = h / img_h
        results.append((CLASS_MAP[cls_name], xc, yc, nw, nh))

    return results

# 샘플 하나로 테스트
test_boxes = parse_label(sample_lbl, 1, 1)  # 정규화 확인용, 실제 img_w/h는 아래 변환 단계에서 사용
print(f"샘플에서 탐지된 객체 수: {len(test_boxes)}")


## 6. YOLO 데이터셋 구조로 변환

Ultralytics YOLO가 요구하는 폴더 구조(`images/train`, `images/val`,
`labels/train`, `labels/val`)로 변환하고, train/val을 9:1로 분리합니다.

In [ ]:
from sklearn.model_selection import train_test_split

OUTPUT_ROOT = Path("./yolo_dataset")
for split in ["train", "val"]:
    (OUTPUT_ROOT / "images" / split).mkdir(parents=True, exist_ok=True)
    (OUTPUT_ROOT / "labels" / split).mkdir(parents=True, exist_ok=True)

train_pairs, val_pairs = train_test_split(pairs, test_size=0.1, random_state=42)
print(f"train: {len(train_pairs):,}개 / val: {len(val_pairs):,}개")


In [ ]:
def convert_and_save(pairs_subset, split, use_symlink=True):
    """use_symlink=True면 이미지를 복사하지 않고 심볼릭 링크로 연결해서
    디스크 용량을 아낍니다. 윈도우 환경이면 use_symlink=False로 바꿔서 복사하세요."""
    skipped = 0
    for img_path, lbl_path in tqdm(pairs_subset, desc=f"{split} 변환중"):
        img = cv2.imread(str(img_path))
        if img is None:
            skipped += 1
            continue
        h, w = img.shape[:2]

        try:
            boxes = parse_label(lbl_path, w, h)
        except KeyError as e:
            skipped += 1
            continue

        if not boxes:
            skipped += 1
            continue

        dst_img = OUTPUT_ROOT / "images" / split / img_path.name
        if not dst_img.exists():
            if use_symlink:
                os.symlink(img_path.resolve(), dst_img)
            else:
                shutil.copy2(img_path, dst_img)

        dst_lbl = OUTPUT_ROOT / "labels" / split / (img_path.stem + ".txt")
        with open(dst_lbl, "w") as f:
            for cls_id, xc, yc, nw, nh in boxes:
                f.write(f"{cls_id} {xc:.6f} {yc:.6f} {nw:.6f} {nh:.6f}\n")

    print(f"[{split}] 스킵된 샘플: {skipped}개")

convert_and_save(train_pairs, "train")
convert_and_save(val_pairs, "val")


## 7. data.yaml 생성

In [ ]:
yaml_lines = [
    f"path: {OUTPUT_ROOT.resolve()}",
    "train: images/train",
    "val: images/val",
    "",
    "names:",
]
for i, name in enumerate(CLASS_NAMES):
    yaml_lines.append(f"  {i}: {name}")

yaml_content = "\n".join(yaml_lines) + "\n"

data_yaml_path = OUTPUT_ROOT / "data.yaml"
with open(data_yaml_path, "w", encoding="utf-8") as f:
    f.write(yaml_content)

print(yaml_content)


## 8. 학습 (COCO 사전학습 가중치 → 파인튜닝)

`yolov8s.pt`는 COCO로 사전학습된 가중치입니다. 처음부터 학습하는 게 아니라
여기서 시작해서 165번 데이터로 파인튜닝합니다.

In [ ]:
model = YOLO("yolov8s.pt")  # COCO 사전학습 가중치 (자동 다운로드)

results = model.train(
    data=str(data_yaml_path),
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,          # GPU(cuda:0) 사용. GPU 여러 개면 [0,1]처럼 리스트로.
    workers=8,
    project="flood_stage1",
    name="vehicle_detector",
    patience=10,        # 10 epoch 동안 성능 향상 없으면 조기 종료
    exist_ok=True,
)


## 9. 검증

In [ ]:
metrics = model.val()
print("mAP50-95:", metrics.box.map)
print("mAP50:   ", metrics.box.map50)
print("클래스별 mAP50:")
for i, name in enumerate(CLASS_NAMES):
    print(f"  {name}: {metrics.box.maps[i]:.4f}")


## 10. 학습된 모델로 빠른 추론 테스트

In [ ]:
best_model_path = f"flood_stage1/vehicle_detector/weights/best.pt"
trained_model = YOLO(best_model_path)

sample_test_img = str(val_pairs[0][0])
result = trained_model.predict(sample_test_img, save=True, conf=0.25)
print("결과 저장 위치:", result[0].save_dir)
